# General Medical Chatbot - By Samiha Azeem

**Lets Connect: Click below**

<a href="https://github.com/samihaazeem2-haru/Short-Term-Stock-Price-Prediction" target="_blank">
  <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png" width="32" />
</a>
&nbsp;&nbsp;&nbsp;
<a href="mailto:samihaazeem2@gmail.com" target="_blank">
  <img src="https://upload.wikimedia.org/wikipedia/commons/4/4e/Gmail_Icon.png" width="32" />
</a>
&nbsp;&nbsp;&nbsp;
<a href="https://www.linkedin.com/in/samiha-azeem-2a7553219/" target="_blank">
  <img src="https://cdn-icons-png.flaticon.com/512/174/174857.png" width="32" />
</a>


In [ ]:
!pip install groq gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 11.0 MB/s eta 0:00:00


In [ ]:
import os

try:
    # Try to get from Colab Secrets first
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = api_key
    print("API key loaded from Colab Secrets!")
except:
    # Fall back to direct input
    print("\nColab Secrets not found.")
    api_key = input("Enter your Groq API key: ").strip()
    if api_key:
        os.environ['GROQ_API_KEY'] = api_key
        print("PI key set successfully!")
    else:
        print("No API key provided.")

API key loaded from Colab Secrets!


In [ ]:
import gradio as gr
from groq import Groq
from typing import List, Tuple
import time

class HealthChatbot:
    """
    A general health query chatbot using LLM with safety filters.
    """

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.client = Groq(api_key=self.api_key)
        self.model = "llama-3.1-8b-instant"

        self.high_risk_keywords = [
            'suicide', 'kill myself', 'end my life', 'self-harm', 'cutting',
            'overdose', 'hurt myself', 'want to die'
        ]

        self.medical_advice_keywords = [
            'should i take', 'what medication', 'prescribe', 'dosage',
            'stop taking', 'diagnosis', 'do i have', 'am i pregnant'
        ]

    def create_system_prompt(self) -> str:
        return """You are a helpful and friendly health information assistant. Your role is to provide general health education and information, NOT to give medical advice or diagnoses.

IMPORTANT GUIDELINES:
1. Be warm, empathetic, and clear in your responses
2. Provide general health information and education only
3. ALWAYS remind users that you cannot replace professional medical advice
4. For symptoms, suggest consulting a healthcare provider without making diagnoses
5. Provide balanced, evidence-based information
6. Use simple, easy-to-understand language
7. If asked about medications, provide general information but emphasize consulting a doctor or pharmacist
8. Never suggest specific treatments or dosages
9. Encourage preventive care and healthy lifestyle choices

RESPONSE FORMAT:
- Start with a friendly acknowledgment
- Provide clear, factual information
- End with a reminder to consult healthcare professionals for personal medical concerns
- Keep responses concise but informative (3-5 paragraphs maximum)

Remember: You're here to educate and inform, not to diagnose or prescribe."""

    def check_safety(self, query: str) -> Tuple[bool, str]:
        query_lower = query.lower()

        for keyword in self.high_risk_keywords:
            if keyword in query_lower:
                crisis_response = """**URGENT: Please Seek Immediate Help**

I'm really concerned about what you're sharing. Please reach out for immediate help:
**International Association for Suicide Prevention**: https://www.iasp.info/resources/Crisis_Centres/

Your life matters, and there are people who want to help you right now. Please contact one of these services immediately or go to your nearest emergency room.

I'm here to provide health information, but this situation needs immediate professional support."""
                return False, crisis_response

        return True, ""

    def enhance_query_with_safety(self, query: str) -> str:
        query_lower = query.lower()

        for keyword in self.medical_advice_keywords:
            if keyword in query_lower:
                return f"""{query}

IMPORTANT CONTEXT: The user's query seems to be seeking medical advice. Please provide general educational information only, and strongly emphasize the need to consult with a healthcare professional, doctor, or pharmacist for personalized medical advice."""

        return query

    def get_response(self, user_query: str, history: List[List[str]]) -> Tuple[str, List[List[str]]]:
        if not user_query.strip():
            return "", history

        is_safe, warning = self.check_safety(user_query)
        if not is_safe:
            history.append([user_query, warning])
            return "", history

        enhanced_query = self.enhance_query_with_safety(user_query)

        messages = [{"role": "system", "content": self.create_system_prompt()}]

        for user_msg, assistant_msg in history:
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

        messages.append({"role": "user", "content": enhanced_query})

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.7,
                max_tokens=800,
                top_p=0.9
            )

            assistant_response = response.choices[0].message.content
            history.append([user_query, assistant_response])

            return "", history

        except Exception as e:
            error_msg = f"I apologize, but I encountered an error: {str(e)}. Please try again or rephrase your question."
            history.append([user_query, error_msg])
            return "", history

print("Chatbot class loaded successfully!")

Chatbot class loaded successfully!


In [ ]:
def create_gradio_interface(api_key: str):
    chatbot = HealthChatbot(api_key=api_key)

    custom_css = """
    .gradio-container {
        font-family: 'Arial', sans-serif;
    }
    """

    examples = [
        ["What causes a sore throat?"],
        ["Is paracetamol safe for children?"],
        ["How can I improve my sleep quality?"],
        ["What are the benefits of drinking water?"],
        ["How do I know if I have a cold or flu?"],
        ["What are common side effects of antibiotics?"],
    ]

    with gr.Blocks(css=custom_css, title="Health Information Chatbot") as demo:
        gr.Markdown(
            """
            General Health Information Chatbot

            Welcome! I'm here to provide **general health information and education**.

            **Important**: I cannot provide medical advice, diagnoses, or treatment recommendations.
            Always consult qualified healthcare professionals for personal medical concerns.

            ### What I Can Help With:
            - General health information and education
            - Common health conditions and symptoms
            - Lifestyle and wellness tips
            - General medication information

            ### Emergency?
            For medical emergencies, please call emergency services (1122 in PK) or visit your nearest emergency room.
            """
        )

        chatbot_interface = gr.Chatbot(
            label="Chat History",
            height=400,
            show_label=True,
            avatar_images=(None, "🤖")
        )

        with gr.Row():
            msg = gr.Textbox(
                label="Your Health Question",
                placeholder="Type your health question here... (e.g., 'What causes a headache?')",
                lines=2,
                scale=4
            )
            submit_btn = gr.Button("Send 📤", scale=1, variant="primary")

        with gr.Row():
            clear_btn = gr.Button("Clear Chat 🗑️")

        gr.Markdown("### 💡 Example Questions:")
        gr.Examples(
            examples=examples,
            inputs=msg,
            label="Click any example to try it"
        )

        gr.Markdown(
            """
            ---
            ### 📋 Disclaimer
            This chatbot provides general health information for educational purposes only. It is not a substitute
            for professional medical advice, diagnosis, or treatment. Always seek the advice of your physician or
            other qualified health provider with any questions you may have regarding a medical condition.

            **Model**: Llama 3.1 8B Instant | **Powered by**: Groq API
            """
        )

        def respond(message, chat_history):
            return chatbot.get_response(message, chat_history)

        msg.submit(respond, [msg, chatbot_interface], [msg, chatbot_interface])
        submit_btn.click(respond, [msg, chatbot_interface], [msg, chatbot_interface])
        clear_btn.click(lambda: [], None, chatbot_interface, queue=False)

    return demo

print("Interface created successfully!")

Interface created successfully!


In [ ]:
# Get API key from environment
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("API key not found! Please run Cell 2 first.")
else:
    print("API key found!")
    print("\nLaunching Gradio interface...")
    print("A public URL will be generated that you can share!")

    # Create and launch the interface
    demo = create_gradio_interface(api_key)
    demo.launch(share=True, debug=True)


API key found!

Launching Gradio interface...
A public URL will be generated that you can share!


/tmp/ipython-input-1455196661.py:40: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_interface = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6a0ebbe8915002099a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6a0ebbe8915002099a.gradio.live


In [ ]:
def test_chatbot():
    """
    Quick test function to verify the chatbot is working.
    Run this cell separately if you want to test without launching UI.
    """
    api_key = os.getenv('GROQ_API_KEY')
    if not api_key:
        print("API key not found!")
        return

    chatbot = HealthChatbot(api_key=api_key)

    test_queries = [
        "What causes a sore throat?",
        "Is paracetamol safe for children?"
    ]

    print("\nTesting Chatbot...")
    print("=" * 60)

    for query in test_queries:
        print(f"\nQuery: {query}")
        _, history = chatbot.get_response(query, [])
        print(f"\nResponse: {history[0][1]}")
        print("\n" + "-" * 60)

# Uncomment below to run quick test
# test_chatbot()